# Stage 1: training the retriever

This notebook trains the retriever (Stage 1), a bi-encoder that turns an address into a 384-number embedding. It then embeds every G-NAF record into an index and measures recall@K on the test set. It follows the two-stage design of Duarte and Oliveira (2023), with two changes:

- Base model: the English model `sentence-transformers/all-MiniLM-L6-v2` replaces multilingual DistilBERT with an MLP-512 head, because the addresses are English only.
- Loss: `MultipleNegativesRankingLoss` replaces the margin contrastive loss, because it also uses the other addresses in each batch as negatives. The paper's three kinds of negative (Section 3.2.2) still supply one negative per positive.

Part 1 (cells 1 to 16) trains the model and builds the index. Part 2 (cells 17 to 24) evaluates them and runs on its own after a kernel restart.

- Reads: `output_synthesis/synthetic_addresses.parquet`, `gnaf.duckdb`, `output/misspellings.tsv`
- Writes: `output_stage1/` (the model, the index, the per-query recall files and two manifests)
- Run after: `2_synthesise_training_data.ipynb`

Training (cell 10) and embedding the index (cell 13, about 84 minutes) are the long cells. With `S1_ALLOW_CACHE=1`, Part 1 reuses the saved model and index.

In [ ]:
# Cell 1: install the packages this notebook imports.
# sentence-transformers 4.1 or later is needed for the trainer used in cell 10.
%pip install -q "sentence-transformers>=4.1" "accelerate>=1.1.0" datasets duckdb pandas pyarrow numpy psutil rapidfuzz tqdm

In [ ]:
# Cell 2: configuration. Every path, model setting and training value is set
# here. S1_DB, S1_SYN, S1_MISS, S1_OUT and S1_MODEL override the paths and the
# base model.

import os, json, time, re
from pathlib import Path

def _env(name, default): return os.environ.get(name, default)

DB_PATH  = Path(_env("S1_DB", "gnaf.duckdb"))
SYN_PATH = next((Path(p) for p in [_env("S1_SYN", ""),
                                   "output_synthesis/synthetic_addresses.parquet"]
                 if p and Path(p).exists()), None)
MISSPELLINGS_TSV = next((Path(p) for p in [_env("S1_MISS", ""), "output/misspellings.tsv",
                                           "misspellings.tsv"] if p and Path(p).exists()), None)
OUT = Path(_env("S1_OUT", "output_stage1")); OUT.mkdir(parents=True, exist_ok=True)

# model: an English sentence-embedding model in place of the paper's
# multilingual DistilBERT with an MLP-512 head, as the addresses are English.
MODEL_NAME = _env("S1_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
EMBED_DIM  = 384
SEED       = 4640
# S1_SMOKE=1 runs a small test of the code (300 positives, 1 epoch).
SMOKE      = _env("S1_SMOKE", "0") == "1"

# training. LR, weight decay and epochs are the values in Duarte and Oliveira
# (2023), Table 1 and Appendix B.
N_TRAIN_POS = 300 if SMOKE else 85_000
HARD_SIM_THRESHOLD = 0.80    # paper: string similarity > 0.8. The paper does
                             # not name the metric, so RapidFuzz JaroWinkler
                             # is used, as elsewhere in the pipeline.
VAL_FRAC = 0.10
EPOCHS   = 1 if SMOKE else 20   # paper's value. Cell 9 projects the cost
LR = 1e-5
WEIGHT_DECAY = 0.01
# MultipleNegativesRankingLoss uses every other positive in the batch as an
# extra negative, so a batch larger than the paper's 16 gives more negatives
# per step.
BATCH   = 64
MAX_LEN = 64
FP16    = False              # MPS has no fp16 training, so True only on CUDA

# index and retrieval
EMBED_BATCH = 256
EMBED_CHUNK = 200_000        # rows per database fetch, encode and write
STORE_DTYPE = "float32"      # cell 8 stops if the index will not fit
RETRIEVE_K  = 50

EMB_NPY   = OUT / "emb_store.npy"
PID_NPY   = OUT / "emb_pids.npy"
PROGRESS  = OUT / "emb_progress.json"

# S1_ALLOW_CACHE=1 makes cell 10 load OUT/model instead of training, and makes
# cells 8 and 16 leave the index and manifest on disk as they are.
ALLOW_CACHE = _env("S1_ALLOW_CACHE", "0") == "1"
# Cell 23 compares this model with the per-query results of an earlier
# retriever, trained on the synthetic set made before the manual label review.
# S1_BASELINE and S1_BASELINE_LABEL override the file and its printed name.
BASELINE_SUBSET = Path(_env("S1_BASELINE",
                            str(Path("results_labels_789_model_85k") / "stage1" / "stage1_recall_by_subset.tsv")))
BASELINE_LABEL = _env("S1_BASELINE_LABEL", "85k model (results_labels_789_model_85k)")

import numpy as np, pandas as pd, duckdb
rng = np.random.default_rng(SEED)
assert SYN_PATH is not None, "no synthetic file found; set S1_SYN"
assert MISSPELLINGS_TSV is not None, "misspellings.tsv not found; set S1_MISS"
print(f"synthetic   : {SYN_PATH}")
print(f"test set    : {MISSPELLINGS_TSV}")
print(f"database    : {DB_PATH}")
print(f"model       : {MODEL_NAME} ({EMBED_DIM}-dim)")
print(f"train pos   : {N_TRAIN_POS:,}   epochs {EPOCHS}   batch {BATCH}")
print(f"smoke mode  : {SMOKE}")
print(f"allow cache : {ALLOW_CACHE}   (S1_ALLOW_CACHE=1 reuses model, index key table, index, manifest)")

synthetic   : output_synthesis/synthetic_addresses.parquet
test set    : output/misspellings.tsv
database    : gnaf.duckdb
model       : sentence-transformers/all-MiniLM-L6-v2 (384-dim)
train pos   : 85,000   epochs 20   batch 64
smoke mode  : False
allow cache : False   (S1_ALLOW_CACHE=1 reuses model, index key table, index, manifest)


In [ ]:
# Cell 3: connect to the G-NAF database and count the rows of the three
# tables used here. The connection is read-write because cell 12 adds a table.
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE INDEX IF NOT EXISTS ix_gnaf_key_pid ON gnaf_key(pid);")
con.execute("SET preserve_insertion_order=false;")
# Seed DuckDB's random() so the hard and very hard negatives drawn in
# cell 6 are the same on every run. The trained model was produced before
# this seed was added, so a re-run may not reproduce it exactly.
con.execute("SELECT setseed(0.4640);")
for t in ["gnaf_key", "gnaf_streets", "gnaf_range"]:
    n = con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
    print(f"  {t:14s} {n:>12,}")
print("connected")

  gnaf_key         20,837,941
  gnaf_streets        530,796
  gnaf_range        1,146,613
connected


In [ ]:
# Cell 4: RENDER, the SQL expression that turns a G-NAF row into an address
# string, for the training positives and the index alike. RJOIN adds the end
# of a number range. RENDER must match the one in 4_train_reranker.ipynb byte
# for byte (cell 17 checks its sha1).
RENDER = r"""regexp_replace(trim(concat_ws(' ',
    CASE WHEN g.number_last IS NOT NULL AND g.number_last <> ''
              AND regexp_matches(k.k, '^[0-9]')
         THEN k.k || '-' || g.number_last ELSE k.k END,
    k.street_n, k.type_n, k.suffix_n, k.locality_n, k.state_n, k.postcode)),
    '\s+', ' ', 'g')"""
RJOIN = "LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k"
print("RENDER ready")

RENDER ready


In [ ]:
# Cell 5: load the synthetic positives and parse each anchor (the correct
# address string) back into street, locality, state and postcode against
# gnaf_streets. The hard and very hard negatives in cell 6 need these parts.

syn = (pd.read_parquet(SYN_PATH) if SYN_PATH.suffix == ".parquet"
       else pd.read_csv(SYN_PATH, sep="\t", dtype=str))
syn = syn.dropna(subset=["dirty", "anchor"]).drop_duplicates("dirty")
if "label" in syn.columns:
    syn = syn[syn.label == "POS"]
take = min(N_TRAIN_POS, len(syn))
pos = syn.sample(take, random_state=SEED).reset_index(drop=True)
print(f"{len(syn):,} synthetic positives available, using {len(pos):,}")
if take < N_TRAIN_POS:
    print(f"  NOTE: asked for {N_TRAIN_POS:,}; the file only has {len(syn):,}.")

anchors = pd.DataFrame({"anchor": pos.anchor.unique()})
con.register("anchors_df", anchors)
con.execute(r"""
CREATE OR REPLACE TEMP TABLE s1_anchor AS
SELECT anchor,
       regexp_replace(anchor, '^(\d+)-\S+ ', '\1 ') AS mk,
       regexp_extract(anchor, '(\S+)$', 1)          AS pc
FROM anchors_df;""")
con.execute(r"""
CREATE OR REPLACE TEMP TABLE s1_comp AS
SELECT a.anchor,
       trim(substr(a.mk, 1, length(a.mk) - length(t.tail))) AS k,
       t.street_n, t.type_n, t.suffix_n, t.locality_n, t.state_n, t.postcode
FROM s1_anchor a
JOIN (
  SELECT *, regexp_replace(trim(concat_ws(' ', street_n, type_n, suffix_n,
              locality_n, state_n, postcode)), '\s+', ' ', 'g') AS tail
  FROM gnaf_streets
) t ON t.postcode = a.pc AND ends_with(a.mk, ' ' || t.tail)
WHERE regexp_matches(trim(substr(a.mk, 1, length(a.mk) - length(t.tail))),
                     '^(LOT \S+|\S+)$')
QUALIFY row_number() OVER (PARTITION BY a.anchor ORDER BY length(t.tail) DESC) = 1;""")
comp = con.sql("SELECT * FROM s1_comp").df()
cov = len(comp) / len(anchors)
print(f"anchor parse coverage: {len(comp):,} of {len(anchors):,} ({cov:.1%})")
assert cov > 0.90, "anchors are not parsing against gnaf_streets; check DB_PATH"
pos = pos.merge(comp, on="anchor", how="inner")
print(f"{len(pos):,} positives with parsed components")

84,971 synthetic positives available, using 84,971
  NOTE: asked for 85,000; the file only has 84,971.


anchor parse coverage: 77,194 of 77,194 (100.0%)
84,971 positives with parsed components


In [ ]:
# Cell 6: mine one negative per positive, its kind drawn at random from the
# three in Duarte and Oliveira (2023), Section 3.2.2:
#   easy       any other record
#   hard       same street name and type, with JaroWinkler similarity to the
#              mutated address (what the model sees) above HARD_SIM_THRESHOLD
#   very hard  same postcode as the true record, a different street
# If the drawn kind has no candidate, the next kind is tried.
from rapidfuzz.distance import JaroWinkler
from collections import defaultdict

POOL_SQL = dict(
    hard=f"""
      SELECT k.street_n, k.type_n, {RENDER} AS cand
      FROM gnaf_key k
      JOIN (SELECT DISTINCT street_n, type_n FROM s1_comp) c USING (street_n, type_n)
      {RJOIN}
      QUALIFY row_number() OVER (PARTITION BY k.street_n, k.type_n
                                 ORDER BY random()) <= 20""",
    very_hard=f"""
      SELECT k.postcode, k.street_n, {RENDER} AS cand
      FROM gnaf_key k
      JOIN (SELECT DISTINCT postcode FROM s1_comp) c USING (postcode)
      {RJOIN}
      QUALIFY row_number() OVER (PARTITION BY k.postcode ORDER BY random()) <= 20""",
    easy=f"""
      SELECT {RENDER} AS cand
      FROM gnaf_key k {RJOIN}
      USING SAMPLE reservoir({max(len(pos) * 2, 3000)} ROWS) REPEATABLE ({SEED})""",
)
pools = {name: con.sql(sql).df() for name, sql in POOL_SQL.items()}
for name, d in pools.items():
    print(f"pool {name:10s}: {len(d):,} rows")

by_hard = defaultdict(list)
for r in pools["hard"].itertuples():  by_hard[(r.street_n, r.type_n)].append(r.cand)
by_vhard = defaultdict(list)
for r in pools["very_hard"].itertuples(): by_vhard[r.postcode].append((r.street_n, r.cand))
easy_pool = pools["easy"].cand.tolist()

CATS = ["easy", "hard", "very_hard"]
rows, missed = [], defaultdict(int)
for r in pos.itertuples():
    for cat in rng.permutation(CATS):
        placed = False
        if cat == "easy":
            for _ in range(5):
                c = easy_pool[int(rng.integers(len(easy_pool)))]
                if c != r.anchor:
                    rows.append((r.dirty, r.anchor, c, "easy")); placed = True; break
        elif cat == "hard":
            opts = [c for c in by_hard.get((r.street_n, r.type_n), []) if c != r.anchor]
            scored = [(c, JaroWinkler.normalized_similarity(r.dirty, c)) for c in opts]
            scored = [(c, s) for c, s in scored if s > HARD_SIM_THRESHOLD]
            if scored:
                rows.append((r.dirty, r.anchor, max(scored, key=lambda x: x[1])[0], "hard"))
                placed = True
        else:
            opts = [c for st, c in by_vhard.get(r.postcode, []) if st != r.street_n]
            if opts:
                rows.append((r.dirty, r.anchor, opts[int(rng.integers(len(opts)))], "very_hard"))
                placed = True
        if placed: break
    else:
        missed["no_negative_any_category"] += 1

triplets = pd.DataFrame(rows, columns=["anchor_text", "positive", "negative", "neg_category"])
print(f"\n{len(triplets):,} triplets from {len(pos):,} positives "
      f"({len(triplets)/len(pos):.1%} coverage; target 100% for a true 1:1 ratio)")
print(triplets.neg_category.value_counts(normalize=True).round(3).to_string())
if missed: print("\nrows with no negative in any category:", dict(missed))

pool hard      : 698,932 rows
pool very_hard : 47,380 rows
pool easy      : 169,942 rows



84,971 triplets from 84,971 positives (100.0% coverage; target 100% for a true 1:1 ratio)
neg_category
easy         0.370
very_hard    0.368
hard         0.263


In [ ]:
# Cell 7: split the triplets into training and validation sets by a hash
# (crc32) of the mutated address, so no mutated address is on both sides.
import zlib
triplets["grp"] = triplets["anchor_text"].map(lambda s: zlib.crc32(s.encode()) % 100)
train_df = triplets[triplets.grp >= VAL_FRAC * 100].drop(columns=["grp", "neg_category"])
val_df   = triplets[triplets.grp <  VAL_FRAC * 100].drop(columns=["grp", "neg_category"])
print(f"train {len(train_df):,} / val {len(val_df):,} (split by query string)")

train 76,371 / val 8,600 (split by query string)


In [ ]:
# Cell 8: check there is disk space for the index before training starts.
# Cell 13 writes the index as a float32 .npy file (22.7 GiB for 15.9 million
# records, plus 0.47 GiB of pids). Under S1_ALLOW_CACHE=1 with a complete
# index on disk, the check is skipped.
import shutil
GB = 1024**3
N_INDEX = con.execute("SELECT count(DISTINCT pid) FROM gnaf_key").fetchone()[0]
need = N_INDEX * EMBED_DIM * 4 + N_INDEX * 32
free = shutil.disk_usage(str(OUT)).free
have_old = con.execute("""SELECT count(*) FROM duckdb_tables()
                          WHERE table_name='s1_embeddings'""").fetchone()[0] > 0

print(f"pids to index : {N_INDEX:,}")
print(f"fp32 index    : {need/GB:6.2f} GiB")
print(f"free disk     : {free/GB:6.2f} GiB")
if have_old:
    print("\n  s1_embeddings still exists in gnaf.duckdb and is probably ~24 GiB.")
    print("  This notebook does not use it. DuckDB never reclaims space on")
    print("  DROP, so dropping it frees nothing on its own. Compact the database once")
    print("  into a fresh file, then delete the old one.")
# with a complete index on disk, cell 13 skips the embed, so no space is needed
_index_done = False
if ALLOW_CACHE and PROGRESS.exists() and EMB_NPY.exists():
    _st = json.loads(PROGRESS.read_text())
    _index_done = _st.get("n_index") == N_INDEX and _st.get("done", 0) >= N_INDEX
if _index_done:
    print("\n[cache] S1_ALLOW_CACHE=1 and the index is complete on disk; the space check "
          "does not apply (cell 13 will skip the embed).")
else:
    assert need < free * 0.92, (
        f"fp32 index needs {need/GB:.1f} GiB but only {free/GB:.1f} GiB is free. "
        "Compact the database or free space. Do not switch to float16: "
        "an fp16 index lost about 4% of the top-50.")
    print("\nOK: fp32 index will fit.")

pids to index : 15,901,232
fp32 index    :  23.22 GiB
free disk     :   6.90 GiB

[cache] S1_ALLOW_CACHE=1 and the index is complete on disk; the space check does not apply (cell 13 will skip the embed).


In [ ]:
# Cell 9: train for 50 steps on the real data, time them, and project the
# full training time. The pilot model is discarded.
import torch
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sentence_transformers.losses import MultipleNegativesRankingLoss
import transformers

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
_tf5  = int(transformers.__version__.split(".")[0]) >= 5
_warm = dict(warmup_steps=0.1) if _tf5 else dict(warmup_ratio=0.1)

def to_ds(d):
    return Dataset.from_pandas(
        d[["anchor_text", "positive", "negative"]].rename(columns={"anchor_text": "anchor"}),
        preserve_index=False)

PILOT_STEPS = 50
_m = SentenceTransformer(MODEL_NAME, device=device); _m.max_seq_length = MAX_LEN
_args = SentenceTransformerTrainingArguments(
    output_dir=str(OUT / "pilot"), max_steps=PILOT_STEPS,
    per_device_train_batch_size=BATCH, learning_rate=LR,
    weight_decay=WEIGHT_DECAY, **_warm, fp16=FP16, bf16=False,
    eval_strategy="no", save_strategy="no", logging_steps=1_000_000,
    seed=SEED, dataloader_num_workers=0, report_to="none")
_t0 = time.time()
SentenceTransformerTrainer(model=_m, args=_args,
                           train_dataset=to_ds(train_df),
                           loss=MultipleNegativesRankingLoss(_m)).train()
_per_step = (time.time() - _t0) / PILOT_STEPS
del _m

steps_per_epoch = int(np.ceil(len(train_df) / BATCH))
total_steps = steps_per_epoch * EPOCHS
print(f"\n  {_per_step:.2f} s per step over {PILOT_STEPS} steps")
print(f"  {steps_per_epoch:,} steps/epoch x {EPOCHS} epochs = {total_steps:,} steps")
print(f"  projected training time: {_per_step*total_steps/3600:.1f} hours")
print(f"  (10 epochs instead: {_per_step*steps_per_epoch*10/3600:.1f} hours)")
print("\nEmbedding the full index afterwards is a separate step of about 1.5 hours.")
print("If the projected time is too long, set EPOCHS=10 in cell 2.")
print("The paper's 20 epochs come from a single BERT study, not from")
print("the paper's own measurement.")

device: mps


Step,Training Loss



  0.39 s per step over 50 steps
  1,194 steps/epoch x 20 epochs = 23,880 steps
  projected training time: 2.6 hours
  (10 epochs instead: 1.3 hours)

Embedding the full index afterwards is a separate step of about 1.5 hours.
If the projected time is too long, set EPOCHS=10 in cell 2.
The paper's 20 epochs come from a single BERT study, not from
the paper's own measurement.


In [ ]:
# Cell 10: fine-tune the retriever and save it to OUT/model. A checkpoint is
# saved every epoch and the last two are kept. With S1_ALLOW_CACHE=1 the saved
# model is loaded instead.
ALLOW_CACHE = _env("S1_ALLOW_CACHE", "0") == "1"
if (OUT / "model").exists() and ALLOW_CACHE:
    model = SentenceTransformer(str(OUT / "model"), device=device)
    print(f"[cache] loaded existing model from {OUT/'model'} (S1_ALLOW_CACHE=1)")
else:
    if (OUT / "model").exists():
        print(f"WARNING: {OUT/'model'} exists and will be overwritten. Make sure")
        print("         any model to keep is copied somewhere else first.")
    model = SentenceTransformer(MODEL_NAME, device=device)
    model.max_seq_length = MAX_LEN
    args = SentenceTransformerTrainingArguments(
        output_dir=str(OUT / "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, **_warm,
        fp16=FP16, bf16=False,
        eval_strategy="epoch",
        save_strategy="epoch", save_total_limit=2,
        logging_steps=50, seed=SEED,
        dataloader_num_workers=0, report_to="none")
    trainer = SentenceTransformerTrainer(
        model=model, args=args,
        train_dataset=to_ds(train_df), eval_dataset=to_ds(val_df),
        loss=MultipleNegativesRankingLoss(model))
    trainer.train()
    model.save_pretrained(str(OUT / "model"))
    print("saved to", OUT / "model")

model.max_seq_length = MAX_LEN
assert model.get_sentence_embedding_dimension() == EMBED_DIM, (
    f"model outputs {model.get_sentence_embedding_dimension()} dims, "
    f"but EMBED_DIM is {EMBED_DIM} in cell 2.")

         any model to keep is copied somewhere else first.


Epoch,Training Loss,Validation Loss
1,0.038092,0.026347
2,0.031459,0.020501
3,0.024153,0.019395
4,0.019533,0.018311
5,0.015810,0.017651
6,0.014028,0.017065
7,0.018392,0.016612
8,0.011781,0.016537
9,0.014254,0.016262
10,0.011067,0.016391


saved to output_stage1/model


In [ ]:
# Cell 11: a quick check before the long embedding step, to catch a model that
# learnt nothing. 300 held-out synthetic queries are searched in a pool of
# 20,000 sampled records plus their true answers. The test-set recall is
# measured in Part 2.
SANITY_POOL_N = 20_000
val_queries = val_df.drop_duplicates("anchor_text").head(300)
pool_rows = con.sql(f"""
    SELECT {RENDER} AS cand FROM gnaf_key k {RJOIN}
    USING SAMPLE reservoir({SANITY_POOL_N} ROWS) REPEATABLE ({SEED + 1})""").df()
pool = (pd.concat([pool_rows, val_queries[["positive"]].rename(columns={"positive": "cand"})],
                  ignore_index=True).drop_duplicates("cand").reset_index(drop=True))
cand_emb = model.encode(pool.cand.tolist(), batch_size=EMBED_BATCH,
                        convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
query_emb = model.encode(val_queries.anchor_text.tolist(), batch_size=EMBED_BATCH,
                         convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
order = np.argsort(-(query_emb @ cand_emb.T), axis=1)[:, :RETRIEVE_K]
cand_idx = {c: i for i, c in enumerate(pool.cand)}
hits = sum(1 for row, ranks in zip(val_queries.itertuples(), order)
           if (i := cand_idx.get(row.positive)) is not None and i in ranks)
print(f"sanity recall@{RETRIEVE_K} on {len(val_queries)} held-out queries, "
      f"{len(pool):,}-row pool: {hits/len(val_queries):.3f}")
print("cheap proxy on synthetic data, not the real Stage 1 number.")

sanity recall@50 on 300 held-out queries, 20,127-row pool: 1.000
cheap proxy on synthetic data, not the real Stage 1 number.


In [ ]:
# Cell 12: choose one gnaf_key row per pid (G-NAF address identifier).
# gnaf_key has 20,837,941 rows for 15,901,232 pids, because some addresses have
# both a street-number key and a LOT key. The street-number key is kept, since
# queries are written with street numbers. A GROUP BY is used rather than a
# window function to keep memory low. A complete existing s1_index_key table is
# reused, because DuckDB does not give back the space a rebuilt table frees.
_n_pids = con.execute("SELECT count(DISTINCT pid) FROM gnaf_key").fetchone()[0]
_have   = con.execute("""SELECT count(*) FROM duckdb_tables()
                         WHERE table_name = 's1_index_key'""").fetchone()[0] > 0
_n_tab  = con.execute("SELECT count(*) FROM s1_index_key").fetchone()[0] if _have else -1
if _have and _n_tab == _n_pids:
    print(f"[cache] s1_index_key already holds {_n_tab:,} rows, one per pid; not rebuilt")
else:
    print(f"[build] s1_index_key ({'missing' if not _have else f'{_n_tab:,} rows, expected {_n_pids:,}'})")
    con.execute("""
    CREATE OR REPLACE TABLE s1_index_key AS
    SELECT pid,
           coalesce(min(CASE WHEN k NOT LIKE 'LOT %' THEN k END), min(k)) AS k
    FROM gnaf_key
    GROUP BY pid;""")
    con.execute("CREATE INDEX IF NOT EXISTS ix_s1_index_key ON s1_index_key(pid);")

d = con.sql("""
SELECT (SELECT count(*) FROM s1_index_key)                                AS pids,
       (SELECT count(*) FROM s1_index_key WHERE k LIKE 'LOT %')           AS lot_only,
       (SELECT count(*) FROM (SELECT pid FROM gnaf_key
          WHERE k NOT LIKE 'LOT %' GROUP BY pid HAVING count(*) > 1))     AS multi_numeric
""").df().iloc[0]
print(f"  distinct pids                  : {int(d.pids):,}")
print(f"  indexed under a LOT key        : {int(d.lot_only):,} "
      f"({d.lot_only/d.pids*100:.2f}%)  <- these have NO street-number form at all")
print(f"  pids with >1 non-LOT key       : {int(d.multi_numeric):,} "
      f"(min() picks one; arbitrary but deterministic)")

# the join back to gnaf_key must give exactly one row per pid
n_join = con.execute("""SELECT count(*) FROM s1_index_key i
                        JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k""").fetchone()[0]
assert n_join == int(d.pids), (
    f"join returns {n_join:,} rows for {int(d.pids):,} pids: a pid has two rows "
    "sharing one key. Add a tiebreak before embedding.")
N_INDEX = int(d.pids)
print(f"  join is 1:1. index size = {N_INDEX:,}")

[cache] s1_index_key already holds 15,901,232 rows, one per pid; not rebuilt


  distinct pids                  : 15,901,232
  indexed under a LOT key        : 263,293 (1.66%)  <- these have NO street-number form at all
  pids with >1 non-LOT key       : 0 (min() picks one; arbitrary but deterministic)


  join is 1:1. index size = 15,901,232


In [ ]:
# Cell 13: embed one rendered string per pid and write it straight into
# OUT/emb_store.npy, with the pids in OUT/emb_pids.npy. Rows are written in pid
# order and OUT/emb_progress.json records how far the cell got, so a re-run
# resumes. It also records the model's modification time, and the cell refuses
# to resume an index written by a different model. Takes about 84 minutes.
_w = sorted((OUT / "model").rglob("*.safetensors")) + sorted((OUT / "model").rglob("*.bin"))
MODEL_MTIME = max(p.stat().st_mtime for p in _w) if _w else 0.0

done, last_pid = 0, ""
if PROGRESS.exists() and EMB_NPY.exists():
    st = json.loads(PROGRESS.read_text())
    if st.get("n_index") == N_INDEX and st.get("dtype") == STORE_DTYPE:
        done, last_pid = st["done"], st["last_pid"]
        if "model_mtime" in st and abs(st["model_mtime"] - MODEL_MTIME) > 1.0:
            raise RuntimeError(
                f"{PROGRESS.name} was written by a DIFFERENT model (sidecar mtime "
                f"{st['model_mtime']}, current {MODEL_MTIME}). Move aside "
                f"{PROGRESS.name}, {EMB_NPY.name} and {PID_NPY.name}, then re-run to "
                "embed from scratch. Refusing to resume or reuse.")
        if done >= N_INDEX:
            print(f"[cache] index complete: {done:,} of {N_INDEX:,} rows already embedded; skipping")
        else:
            print(f"resuming: {done:,} of {N_INDEX:,} already written")
        if "model_mtime" not in st:
            print("  NOTE: this sidecar predates the model_mtime check, so it cannot say")
            print("        which model made the vectors. Cell 19 measures that directly.")

t0 = time.time()
if done < N_INDEX:
    mode = "r+" if (done and EMB_NPY.exists()) else "w+"
    E = np.lib.format.open_memmap(EMB_NPY, mode=mode, dtype=np.float32, shape=(N_INDEX, EMBED_DIM))
    Pm = np.lib.format.open_memmap(PID_NPY, mode=mode, dtype="S32", shape=(N_INDEX,))
while done < N_INDEX:
    t_a = time.time()
    chunk = con.execute(f"""
        SELECT i.pid, {RENDER} AS cand
        FROM s1_index_key i
        JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k
        {RJOIN}
        WHERE i.pid > ?
        ORDER BY i.pid
        LIMIT {EMBED_CHUNK}""", [last_pid]).df()
    if chunk.empty:
        break
    t_b = time.time()
    emb = model.encode(chunk.cand.tolist(), batch_size=EMBED_BATCH,
                       convert_to_numpy=True, normalize_embeddings=True,
                       show_progress_bar=False).astype(np.float32)
    t_c = time.time()
    n = len(chunk)
    E[done:done + n]  = emb
    Pm[done:done + n] = np.asarray(chunk.pid.tolist(), dtype="S32")
    E.flush(); Pm.flush()
    done += n; last_pid = chunk.pid.iloc[-1]
    PROGRESS.write_text(json.dumps({"done": done, "last_pid": last_pid,
                                    "n_index": N_INDEX, "dtype": STORE_DTYPE,
                                    "model_mtime": MODEL_MTIME}))
    el = time.time() - t0
    rate = done / max(el, 1e-9)
    print(f"  {done:,}/{N_INDEX:,}  query {t_b-t_a:5.1f}s | encode {t_c-t_b:5.1f}s | "
          f"write {time.time()-t_c:4.1f}s  (~{(N_INDEX-done)/max(rate,1e-9)/60:.0f} min left)")

if "E" in globals():
    del E, Pm
assert done == N_INDEX, f"wrote {done:,} rows but expected {N_INDEX:,}"
print(f"\ndone: {done:,} rows in {(time.time()-t0)/60:.1f} min")
print(f"  {EMB_NPY}  {EMB_NPY.stat().st_size/GB:.1f} GiB (float32)")
print(f"  {PID_NPY}  {PID_NPY.stat().st_size/GB:.2f} GiB")

_p = np.asarray(np.load(EMB_NPY, mmap_mode="r")[:2000]).astype(np.float32)
_ap = np.load(PID_NPY)
print(f"  norm of first 2,000 rows : mean {np.linalg.norm(_p, axis=1).mean():.4f} (want ~1.0000)")
print(f"  pids unique              : {len(np.unique(_ap)) == len(_ap)}")
print(f"  pids sorted              : {bool((_ap[:-1] <= _ap[1:]).all())}")

  200,000/15,901,232  query   5.3s | encode  54.5s | write  0.2s  (~79 min left)


  400,000/15,901,232  query   4.9s | encode  55.2s | write  0.3s  (~78 min left)


  600,000/15,901,232  query   4.9s | encode  54.9s | write  0.2s  (~77 min left)


  800,000/15,901,232  query   4.6s | encode  55.5s | write  0.2s  (~76 min left)


  1,000,000/15,901,232  query   4.7s | encode  55.6s | write  0.3s  (~75 min left)


...

  15,400,000/15,901,232  query   0.4s | encode  64.6s | write  0.9s  (~3 min left)


  15,600,000/15,901,232  query   0.4s | encode  63.7s | write  0.6s  (~2 min left)


  15,800,000/15,901,232  query   0.3s | encode  63.4s | write  0.6s  (~1 min left)


  15,901,232/15,901,232  query   0.2s | encode  32.0s | write  0.3s  (~0 min left)

done: 15,901,232 rows in 83.6 min
  output_stage1/emb_store.npy  22.7 GiB (float32)
  output_stage1/emb_pids.npy  0.47 GiB
  norm of first 2,000 rows : mean 1.0000 (want ~1.0000)


  pids unique              : True
  pids sorted              : True


In [ ]:
# Cell 14: for 200 pids that have both a street-number key and a LOT key,
# encode both forms and compare each with the stored vector. The form with
# cosine near 1.0 is the one in the index. After cell 12 it should always be
# the street-number form.
dual = con.sql("""
    SELECT pid FROM gnaf_key
    GROUP BY pid
    HAVING count(*) > 1
       AND max(CASE WHEN k LIKE 'LOT %' THEN 1 ELSE 0 END) = 1
       AND min(CASE WHEN k LIKE 'LOT %' THEN 1 ELSE 0 END) = 0
    LIMIT 200""").df()
print(f"{len(dual)} dual-form pids sampled")
con.register("dual_df", dual)
forms = con.sql(f"""
    SELECT k.pid, k.k, {RENDER} AS s
    FROM dual_df d JOIN gnaf_key k ON k.pid = d.pid
    {RJOIN}""").df()
forms["is_lot"] = forms.k.str.upper().str.startswith("LOT")

A = np.load(EMB_NPY, mmap_mode="r"); PA = np.load(PID_NPY)
loc = np.searchsorted(PA, np.array([p.encode() for p in dual.pid]))
ok  = (loc < len(PA)) & (PA[np.clip(loc, 0, len(PA)-1)] ==
                         np.array([p.encode() for p in dual.pid]))
stored = {p: np.asarray(A[i]).astype(np.float32)
          for p, i, o in zip(dual.pid, loc, ok) if o}
print(f"  {len(stored)} of {len(dual)} found in the index")

enc = model.encode(forms.s.tolist(), batch_size=EMBED_BATCH, convert_to_numpy=True,
                   normalize_embeddings=True, show_progress_bar=False)
forms["sim_to_stored"] = [float(v @ stored[p]) if p in stored else np.nan
                          for p, v in zip(forms.pid, enc)]
best = (forms.dropna(subset=["sim_to_stored"])
             .sort_values("sim_to_stored", ascending=False)
             .groupby("pid").first())
share_lot = float(best.is_lot.mean())
print(f"\n  indexed under the LOT form     : {share_lot*100:5.1f}%")
print(f"  indexed under the number form  : {(1-share_lot)*100:5.1f}%")
print(f"  median cosine to the winner    : {best.sim_to_stored.median():.4f} (want ~1.0)")
if share_lot > 0.01:
    print("\n  FAIL: some pids are indexed under their LOT string. Cell 12 should")
    print("  have made this impossible; re-run cells 12 and 13.")
    print(best[best.is_lot].head(5)[["k", "s", "sim_to_stored"]].to_string())
else:
    print("\n  PASS: every dual-form pid is indexed under its street-number string.")

200 dual-form pids sampled


  200 of 200 found in the index



  indexed under the LOT form     :   0.0%
  indexed under the number form  : 100.0%
  median cosine to the winner    : 1.0000 (want ~1.0)

  PASS: every dual-form pid is indexed under its street-number string.


In [ ]:
# Cell 15: RENDER drops unit and level, so all units in one building share one
# string and one embedding. This cell measures how often that happens and
# whether the pids behind one string sit at the same latitude and longitude.
# If they do, scoring recall on distinct strings is justified. The exact count
# runs only over the test set's postcodes.
g = con.sql("""
    SELECT count(*) AS rows,
           approx_count_distinct(k.ADDRESS_LABEL) AS approx_labels
    FROM s1_index_key i JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k""").df().iloc[0]
print("WHOLE INDEX")
print(f"  indexed rows                    : {int(g.rows):,}")
print(f"  approx distinct ADDRESS_LABELs  : {int(g.approx_labels):,}")
print("  (labels keep unit detail, so this is a loose lower bound on how much")
print("   RENDER collapses; the exact figure for the test population is below)\n")

_ms = pd.read_csv(MISSPELLINGS_TSV, sep="\t", dtype=str).fillna("")
_pcs = sorted({m[-1] for m in
               (re.findall(r"\b\d{4}\b", s) for s in _ms.matched) if m})
con.register("pc_df", pd.DataFrame({"postcode": _pcs}))
print(f"TEST-SET POPULATION ({len(_pcs)} postcodes)")

coll = con.sql(f"""
WITH idx AS (
    SELECT k.pid, {RENDER} AS s
    FROM s1_index_key i
    JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k
    {RJOIN}
    JOIN pc_df p ON p.postcode = k.postcode
), geo AS (
    SELECT idx.s, idx.pid,
           try_cast(c.LATITUDE AS DOUBLE)  AS lat,
           try_cast(c.LONGITUDE AS DOUBLE) AS lon
    FROM idx LEFT JOIN core c ON c.ADDRESS_DETAIL_PID = idx.pid
)
SELECT s, count(*) AS n_pids,
       count(lat) AS n_geo,
       (max(lat) - min(lat)) * 111320.0                       AS d_lat_m,
       (max(lon) - min(lon)) * 111320.0 * cos(radians(avg(lat))) AS d_lon_m
FROM geo GROUP BY s HAVING count(*) > 1""").df()
coll["spread_m"] = np.hypot(coll.d_lat_m.fillna(0), coll.d_lon_m.fillna(0))

tot = con.sql(f"""
    SELECT count(*) AS rows, count(DISTINCT {RENDER}) AS distinct_s
    FROM s1_index_key i JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k
    {RJOIN} JOIN pc_df p ON p.postcode = k.postcode""").df().iloc[0]
print(f"  indexed rows                    : {int(tot.rows):,}")
print(f"  distinct rendered strings       : {int(tot.distinct_s):,}")
print(f"  collapse factor                 : {tot.rows/max(tot.distinct_s,1):.2f} pids per string")
print(f"  strings held by >1 pid          : {len(coll):,}")
if len(coll):
    print(f"  largest collision group         : {int(coll.n_pids.max()):,} pids")
    q = coll.spread_m.quantile([.5, .9, .99, 1.0]).round(1)
    print(f"\n  COORDINATE SPREAD within a collision group (metres):")
    print(f"    median {q.loc[0.5]}   p90 {q.loc[0.9]}   p99 {q.loc[0.99]}   max {q.loc[1.0]}")
    print(f"    groups where every pid is within 50 m : "
          f"{(coll.spread_m <= 50).mean()*100:.1f}%")
    print("\n  10 widest groups (these are the ones that are NOT interchangeable):")
    print(coll.nlargest(10, "spread_m")[["s", "n_pids", "spread_m"]]
              .to_string(index=False, max_colwidth=52))
print("""
  If nearly all groups are tight, the pids behind one string are the same
  building. Recall is then scored on distinct rendered addresses rather than
  on raw pids, and candidates are deduplicated before they reach Stage 2, so
  the cross-encoder is not handed K copies of one address. If the spread is
  wide, recall should be scored on pids.""")

WHOLE INDEX
  indexed rows                    : 15,901,232
  approx distinct ADDRESS_LABELs  : 18,022,405
  (labels keep unit detail, so this is a loose lower bound on how much
   RENDER collapses; the exact figure for the test population is below)

TEST-SET POPULATION (407 postcodes)


  indexed rows                    : 5,940,247
  distinct rendered strings       : 3,430,739
  collapse factor                 : 1.73 pids per string
  strings held by >1 pid          : 236,264
  largest collision group         : 2,275 pids

  COORDINATE SPREAD within a collision group (metres):
    median 0.0   p90 37.5   p99 148.3   max 13058.3
    groups where every pid is within 50 m : 93.8%

  10 widest groups (these are the ones that are NOT interchangeable):
                                                   s  n_pids     spread_m
                    5 MCLEAN STREET BEVERLEY SA 5009      13 13058.281833
LOT 70 SOUTH STRADBROKE ISLAND SOUTH STRADBROKE Q...       2  9175.565517
LOT 68 SOUTH STRADBROKE ISLAND SOUTH STRADBROKE Q...       2  8825.877817
LOT 21 SOUTH STRADBROKE ISLAND SOUTH STRADBROKE Q...       2  8620.840353
LOT 15 SOUTH STRADBROKE ISLAND SOUTH STRADBROKE Q...       2  8482.595063
LOT 13 SOUTH STRADBROKE ISLAND SOUTH STRADBROKE Q...       2  8451.345613
LOT 12 SOUTH 

In [ ]:
# Cell 16: write OUT/run_manifest.json (settings, sizes, time and git hash),
# so a smoke run cannot be mistaken for a full one. Under S1_ALLOW_CACHE=1 the
# existing manifest is printed and left unchanged.
import subprocess
manifest = dict(
    model_name=MODEL_NAME, embed_dim=EMBED_DIM, seed=SEED, smoke=SMOKE,
    n_train_pos=N_TRAIN_POS, n_triplets=len(triplets),
    epochs=EPOCHS, batch=BATCH, lr=LR, weight_decay=WEIGHT_DECAY,
    max_len=MAX_LEN, retrieve_k=RETRIEVE_K,
    index_rows=N_INDEX, index_dtype=STORE_DTYPE,
    synthetic_file=str(SYN_PATH), test_file=str(MISSPELLINGS_TSV),
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
try:
    manifest["git_hash"] = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    manifest["git_hash"] = None
if ALLOW_CACHE and (OUT / "run_manifest.json").exists():
    print(f"[cache] S1_ALLOW_CACHE=1: leaving the existing {OUT/'run_manifest.json'} untouched:")
    print((OUT / "run_manifest.json").read_text())
else:
    (OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest, indent=2))
    print(f"\nwrote {OUT/'run_manifest.json'}")
print("\nNEXT: Part 2 below evaluates this model and index on the real test set.")

{
  "model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "embed_dim": 384,
  "seed": 4640,
  "smoke": false,
  "n_train_pos": 85000,
  "n_triplets": 84971,
  "epochs": 20,
  "batch": 64,
  "lr": 1e-05,
  "weight_decay": 0.01,
  "max_len": 64,
  "retrieve_k": 50,
  "index_rows": 15901232,
  "index_dtype": "float32",
  "synthetic_file": "output_synthesis/synthetic_addresses.parquet",
  "test_file": "output/misspellings.tsv",
  "timestamp": "2026-09-08 00:08:44",
  "git_hash": "098a492ef0c60226e5d4a0c53329198f44758769"
}

wrote output_stage1/run_manifest.json

NEXT: Part 2 below evaluates this model and index on the real test set.


## Part 2: evaluation on the test set

Part 2 runs on its own after a kernel restart. Cell 17 rebuilds the connection, the model and RENDER if Part 1 is not in memory. Nothing here trains or embeds. Cell 20 is the slow cell, at about 2 minutes for one pass over the 22.7 GiB index with all queries at once.

In [ ]:
# Cell 17: set up Part 2 so it can run on its own. If Part 1 is in memory its
# objects are reused. Otherwise the configuration, the database connection,
# the trained model and RENDER are rebuilt from disk. RENDER's sha1 is checked
# either way.
import os, re, time, shutil, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd, duckdb, torch
from sentence_transformers import SentenceTransformer

def _env(name, default): return os.environ.get(name, default)

_reuse = all(n in globals() for n in ["con", "model", "RENDER", "RJOIN", "OUT", "MISSPELLINGS_TSV"])
if _reuse:
    print("Part 1 is in memory: reusing con, model, RENDER, RJOIN and the config")
else:
    print("fresh kernel: rebuilding the Part 2 bootstrap from disk")
    DB_PATH  = Path(_env("S1_DB", "gnaf.duckdb"))
    OUT      = Path(_env("S1_OUT", "output_stage1"))
    MISSPELLINGS_TSV = next((Path(p) for p in [_env("S1_MISS", ""), "output/misspellings.tsv",
                                               "misspellings.tsv"] if p and Path(p).exists()), None)
    MODEL_NAME  = _env("S1_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
    EMBED_DIM   = 384
    RETRIEVE_K  = 50
    EMBED_BATCH = 256
    SEED        = 4640
    MAX_LEN     = 64
    ALLOW_CACHE = _env("S1_ALLOW_CACHE", "0") == "1"
    BASELINE_SUBSET = Path(_env("S1_BASELINE",
                                str(Path("results_labels_789_model_85k") / "stage1" / "stage1_recall_by_subset.tsv")))
    # the baseline for cell 23, as in cell 2
    BASELINE_LABEL = _env("S1_BASELINE_LABEL", "85k model (results_labels_789_model_85k)")

    assert (OUT / "model").exists(), f"no cached model at {OUT/'model'}; run Part 1 first"
    assert MISSPELLINGS_TSV is not None, "misspellings.tsv not found; set S1_MISS"

    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE INDEX IF NOT EXISTS ix_gnaf_key_pid ON gnaf_key(pid);")

    device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    model = SentenceTransformer(str(OUT / "model"), device=device)

    # RENDER, copied from cell 4 (its sha1 is checked below)
    RENDER = r"""regexp_replace(trim(concat_ws(' ',
    CASE WHEN g.number_last IS NOT NULL AND g.number_last <> ''
              AND regexp_matches(k.k, '^[0-9]')
         THEN k.k || '-' || g.number_last ELSE k.k END,
    k.street_n, k.type_n, k.suffix_n, k.locality_n, k.state_n, k.postcode)),
    '\s+', ' ', 'g')"""
    RJOIN = "LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k"

GB = 1024**3
_H = hashlib.sha1(RENDER.encode()).hexdigest()
assert _H == "cc573253b3ead7dddebc8393697909d49f27eaa1", (
    f"RENDER has drifted (sha1 {_H}); it must match Part 1's cell 4 byte for byte")
if model.max_seq_length != MAX_LEN:
    print(f"  WARNING: reloaded max_seq_length is {model.max_seq_length}, forcing to {MAX_LEN} "
          "to match the index build")
    model.max_seq_length = MAX_LEN

# recorded by cell 24: which test set and which model weights this run used
TEST_SHA1 = hashlib.sha1(Path(MISSPELLINGS_TSV).read_bytes()).hexdigest()
_w = sorted((OUT / "model").rglob("*.safetensors")) + sorted((OUT / "model").rglob("*.bin"))
MODEL_MTIME = max(p.stat().st_mtime for p in _w) if _w else 0.0

print(f"device     : {device}")
print(f"database   : {DB_PATH}")
print(f"model      : {OUT/'model'} ({model.get_sentence_embedding_dimension()}-dim, mtime {MODEL_MTIME:.0f})")
print(f"test set   : {MISSPELLINGS_TSV} (sha1 {TEST_SHA1[:12]}...)")
print(f"baseline   : {BASELINE_SUBSET}")
assert model.get_sentence_embedding_dimension() == EMBED_DIM
print("Part 2 ready")

fresh kernel: rebuilding the Part 2 bootstrap from disk


device     : mps
database   : gnaf.duckdb
model      : output_stage1/model (384-dim, mtime 1788792286)
test set   : output/misspellings.tsv (sha1 d48fdfe227fd...)
baseline   : results_labels_789_model_85k/stage1/stage1_recall_by_subset.tsv
Part 2 ready


In [ ]:
# Cell 18: open the index files and define batched_topk(), which scores all
# queries against every stored embedding in one pass over the file and keeps a
# running top K per query. Peak memory is about 1.2 GB for 789 queries (a block
# of 250,000 vectors plus the similarity matrix).
import json
EMB_NPY = OUT / "emb_store.npy"
PID_NPY = OUT / "emb_pids.npy"
assert EMB_NPY.exists() and PID_NPY.exists(), (
    f"{EMB_NPY} or {PID_NPY} missing; run cell 13 of the training notebook first")

_E = np.load(EMB_NPY, mmap_mode="r")
_P = np.load(PID_NPY)
STORE_DTYPE = _E.dtype
assert _E.shape[0] == len(_P), (
    f"index has {_E.shape[0]:,} vectors but {len(_P):,} pids: the two files "
    "disagree and every retrieved pid would be the wrong one. Re-run cell 13.")
assert _E.shape[1] == EMBED_DIM, f"index is {_E.shape[1]}-dim, expected {EMBED_DIM}"

print(f"index      : {_E.shape[0]:,} x {_E.shape[1]}  ({np.dtype(STORE_DTYPE).name})")
print(f"pids unique: {len(np.unique(_P)) == len(_P)}   sorted: {bool((_P[:-1] <= _P[1:]).all())}")
if STORE_DTYPE != np.float32:
    print(f"  NOTE: index is {np.dtype(STORE_DTYPE).name}, not float32. The last "
          "fp16 index lost about 4% of the top-50.")

# check the index against run_manifest.json, so a stale index is caught
_mf = OUT / "run_manifest.json"
if _mf.exists():
    m = json.loads(_mf.read_text())
    print(f"manifest   : {m['index_rows']:,} rows, {m['index_dtype']}, "
          f"{m['n_train_pos']:,} train pos, {m['epochs']} epochs, "
          f"built {m['timestamp']}")
    assert m["index_rows"] == _E.shape[0], (
        "the manifest and the .npy disagree on row count; one of them is stale")
del _E, _P


def batched_topk(queries, k=RETRIEVE_K, chunk_rows=250_000, verbose=True):
    E = np.load(EMB_NPY, mmap_mode="r")
    P = np.load(PID_NPY)
    N = E.shape[0]

    Q = model.encode(list(queries), batch_size=EMBED_BATCH, convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=verbose).astype(np.float32)
    nq = len(Q)
    run_sim = np.full((nq, k), -np.inf, dtype=np.float32)
    run_idx = np.zeros((nq, k), dtype=np.int64)

    t0 = time.time()
    for s in range(0, N, chunk_rows):
        block = np.asarray(E[s:s+chunk_rows]).astype(np.float32, copy=False)
        sims = Q @ block.T                                    # (nq, chunk)

        kk = min(k, sims.shape[1])
        part = np.argpartition(-sims, kk-1, axis=1)[:, :kk]
        cand_sim = np.take_along_axis(sims, part, axis=1)
        cand_idx = part.astype(np.int64) + s

        all_sim = np.concatenate([run_sim, cand_sim], axis=1)
        all_idx = np.concatenate([run_idx, cand_idx], axis=1)
        sel = np.argpartition(-all_sim, k-1, axis=1)[:, :k]
        run_sim = np.take_along_axis(all_sim, sel, axis=1)
        run_idx = np.take_along_axis(all_idx, sel, axis=1)

        if verbose and (s // chunk_rows) % 20 == 0:
            frac = min(s + chunk_rows, N) / N; el = time.time() - t0
            print(f"  {min(s+chunk_rows,N):,}/{N:,}  {el:6.0f}s, "
                  f"~{el/max(frac,1e-9)-el:5.0f}s left")

    order = np.argsort(-run_sim, axis=1)
    run_sim = np.take_along_axis(run_sim, order, axis=1)
    run_idx = np.take_along_axis(run_idx, order, axis=1)
    if verbose:
        el = time.time() - t0
        print(f"done in {el/60:.1f} min ({el/max(nq,1)*1000:.0f} ms per query, amortised)")
    return P[run_idx], run_sim

print("\nbatched_topk ready")

index      : 15,901,232 x 384  (float32)


pids unique: True   sorted: True
manifest   : 15,901,232 rows, float32, 85,000 train pos, 20 epochs, built 2026-09-08 00:08:44

batched_topk ready


In [ ]:
# Cell 19: check that the stored vectors are what the model produces now.
# For the first and last 200 pids, each string is encoded twice and compared
# with itself (is the encoder deterministic?) and with the stored vector (does
# the index belong to this model?). Both should be about 1.0.
_probe = con.sql(f"""
    SELECT i.pid, {RENDER} AS s
    FROM s1_index_key i
    JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k
    {RJOIN}
    ORDER BY i.pid
    LIMIT 200""").df()
_late = con.sql(f"""
    SELECT i.pid, {RENDER} AS s
    FROM s1_index_key i
    JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k
    {RJOIN}
    ORDER BY i.pid DESC
    LIMIT 200""").df()

E = np.load(EMB_NPY, mmap_mode="r"); P = np.load(PID_NPY)

def _check(df, label):
    want = np.array([p.encode() for p in df.pid])
    loc  = np.searchsorted(P, want)
    ok   = (loc < len(P)) & (P[np.clip(loc, 0, len(P)-1)] == want)
    assert ok.all(), f"{label}: some pids are not in the index at all"
    stored = np.asarray(E[loc]).astype(np.float32)

    a = model.encode(df.s.tolist(), batch_size=EMBED_BATCH, convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=False)
    b = model.encode(df.s.tolist(), batch_size=EMBED_BATCH, convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=False)
    enc_enc    = (a * b).sum(1)
    enc_stored = (a * stored).sum(1)
    print(f"{label:22s}  encode-vs-encode  median {np.median(enc_enc):.5f}  "
          f"min {enc_enc.min():.5f}")
    print(f"{'':22s}  encode-vs-stored  median {np.median(enc_stored):.5f}  "
          f"min {enc_stored.min():.5f}  below 0.99: "
          f"{(enc_stored < 0.99).mean()*100:.1f}%")
    return enc_stored

# the first and last pids sit at opposite ends of the index, so an index written
# in two sittings with different weights shows as a gap between them
_a = _check(_probe, "earliest 200 pids")
_b = _check(_late,  "latest 200 pids")
print(f"\ngap between the two halves of the index: "
      f"{abs(np.median(_a) - np.median(_b)):.5f} (want ~0)")
del E, P

earliest 200 pids       encode-vs-encode  median 1.00000  min 1.00000
                        encode-vs-stored  median 1.00000  min 1.00000  below 0.99: 0.0%
latest 200 pids         encode-vs-encode  median 1.00000  min 1.00000
                        encode-vs-stored  median 1.00000  min 1.00000  below 0.99: 0.0%

gap between the two halves of the index: 0.00000 (want ~0)


In [ ]:
# Cell 20: recall curve to K=500 on the matchable test rows. A hit counts if
# the retrieved record is the true record (raw pid) or a G-NAF alias of it
# (same PRINCIPAL_PID). Rows the manual label review marked as excluded are
# dropped and listed. Writes OUT/stage1_recall_per_query.tsv. The scan reads
# the index once whatever K is, so K=500 costs about the same as K=50.
EVAL_K = 500

full = pd.read_csv(MISSPELLINGS_TSV, sep="\t", dtype=str).fillna("")
if "adjudication" in full.columns:
    _exc = full.adjudication.str.contains("excluded", regex=False)
    print(f"{int(_exc.sum())} rows excluded by adjudication (skipped):")
    for _a in full.address[_exc]:
        print(f"    {_a}")
    full = full[~_exc].reset_index(drop=True)
else:
    print("no adjudication column in this test set; nothing excluded")
no_rec = full.reason.str.contains("Cannot be matched")
no_num = full.reason.str.contains("Street number does not exist")
mis = full[~(no_rec | no_num)].reset_index(drop=True)
mis["rid"] = mis.index
print(f"{len(full):,} harvested rows | {len(mis):,} matchable "
      f"({int(no_rec.sum())} no_record, {int(no_num.sum())} no_number excluded)")

def norm(s): return re.sub(r"\s+", " ", str(s).upper().replace(",", " ")).strip()
mis["q"] = mis.address.map(norm)
mis["m"] = mis.matched.map(norm)

con.register("mis_df", mis[["rid", "m"]])
truth = con.sql(f"""
    SELECT m.rid, k.pid AS truth_pid, k.postcode AS truth_postcode
    FROM mis_df m
    JOIN gnaf_key k
      ON regexp_replace(trim(replace(upper(k.ADDRESS_LABEL), ',', ' ')), '\\s+', ' ', 'g') = m.m
    {RJOIN}
    QUALIFY row_number() OVER (PARTITION BY m.rid
        ORDER BY CASE WHEN k.k LIKE 'LOT %' THEN 1 ELSE 0 END, k.pid) = 1
""").df()
test = mis.merge(truth, on="rid", how="inner")
print(f"truth join coverage: {len(test):,} of {len(mis):,} ({len(test)/len(mis):.1%})")

# retrieve the top 500 once
pids, sims = batched_topk(test.q.tolist(), k=EVAL_K)

# map each pid seen here to its principal pid. If gnaf_key has no
# PRINCIPAL_PID column, the raw pid is used
cols = set(con.execute("DESCRIBE gnaf_key").df().column_name.str.upper())
HAS_PRINCIPAL = "PRINCIPAL_PID" in cols
if HAS_PRINCIPAL:
    need = set(test.truth_pid.tolist())
    for row in pids:
        need.update(b.decode() for b in row)
    con.register("need_df", pd.DataFrame({"pid": list(need)}))
    pm = con.execute("""
        SELECT n.pid,
               COALESCE(NULLIF(CAST(k.PRINCIPAL_PID AS VARCHAR), ''), n.pid) AS principal
        FROM need_df n JOIN gnaf_key k ON k.pid = n.pid
        QUALIFY row_number() OVER (PARTITION BY n.pid ORDER BY k.pid) = 1
    """).df()
    P2P = dict(zip(pm.pid, pm.principal))
    print(f"principal-PID map built for {len(P2P):,} pids")
else:
    P2P = {}
    print("gnaf_key has no PRINCIPAL_PID column; principal column will equal raw")
def princ(p): return P2P.get(p, p)

# rank of the true record in each list, by raw pid and by principal pid
ranks_raw, ranks_prn = [], []
for tp, row in zip(test.truth_pid, pids):
    dec = [b.decode() for b in row]
    ranks_raw.append(dec.index(tp) + 1 if tp in dec else -1)
    tp_pr = princ(tp)
    dec_pr = [princ(p) for p in dec]
    ranks_prn.append(dec_pr.index(tp_pr) + 1 if tp_pr in dec_pr else -1)
ranks_raw = np.array(ranks_raw); ranks_prn = np.array(ranks_prn)

def rec_at(r, k): return ((r > 0) & (r <= k)).mean()
def wilson(p, n, z=1.96):
    d = 1 + z*z/n; c = p + z*z/(2*n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c - h)/d, (c + h)/d

print(f"\n  K     raw-pid   principal-normalised")
for kk in [1, 5, 10, 20, 50, 100, 200, 500]:
    print(f"   {kk:<4d}  {rec_at(ranks_raw,kk):.4f}    {rec_at(ranks_prn,kk):.4f}")

p50 = rec_at(ranks_prn, 50); n = len(ranks_prn); lo, hi = wilson(p50, n)
print(f"\n*** STAGE 1 RECALL@50 (principal-normalised): {p50:.4f}  [{lo:.3f}, {hi:.3f}] ***")
print(f"    alias fix moved recall@50 from {rec_at(ranks_raw,50):.4f} to {p50:.4f}")

eval_ = pd.DataFrame({"address": test.address, "truth_pid": test.truth_pid,
                      "rank_raw": ranks_raw, "rank_principal": ranks_prn,
                      "top_sim": sims[:, 0]})
eval_.to_csv(OUT / "stage1_recall_per_query.tsv", sep="\t", index=False)

# share of queries whose written postcode starts with the same digit as the
# true record's postcode: the most that routing by that digit could keep
_pc = re.compile(r"(\d{4})\s*$")
q_digit = test.q.map(lambda s: (_pc.search(s.strip()).group(1)[0] if _pc.search(s.strip()) else None))
t_digit = test.truth_postcode.astype(str).str[0]
no_pc = q_digit.isna(); mismatch = (~no_pc) & (q_digit != t_digit)
print(f"\npostcode-digit routing ceiling: {1 - (no_pc | mismatch).mean():.4f} "
      f"({int(mismatch.sum())} of {len(test)} written digits disagree with truth)")

0 rows excluded by adjudication (skipped):
950 harvested rows | 789 matchable (32 no_record, 129 no_number excluded)


truth join coverage: 789 of 789 (100.0%)


  250,000/15,901,232       2s, ~  137s left


  5,250,000/15,901,232      42s, ~   85s left


  10,250,000/15,901,232      82s, ~   45s left


  15,250,000/15,901,232     124s, ~    5s left


done in 2.2 min (164 ms per query, amortised)


principal-PID map built for 364,569 pids

  K     raw-pid   principal-normalised
   1     0.5336    0.5463
   5     0.6806    0.6882
   10    0.7490    0.7541
   20    0.8162    0.8213
   50    0.9100    0.9125
   100   0.9430    0.9455
   200   0.9632    0.9645
   500   0.9670    0.9683

*** STAGE 1 RECALL@50 (principal-normalised): 0.9125  [0.891, 0.930] ***
    alias fix moved recall@50 from 0.9100 to 0.9125

postcode-digit routing ceiling: 0.9848 (12 of 789 written digits disagree with truth)


In [ ]:
# Cell 21: for each test query, find the similarity of its true record and
# compare it with the 50th and 500th retrieved similarity. This separates
# misses that a deeper K recovers, misses even at K=500, and true records
# that are not in the index at all.
allpids = np.load(PID_NPY)
order = np.argsort(allpids, kind="stable")
sorted_pids = allpids[order]
want = np.array([tp.encode() for tp in test.truth_pid])
loc = np.clip(np.searchsorted(sorted_pids, want), 0, len(sorted_pids) - 1)
found = sorted_pids[loc] == want
row_of_truth = np.where(found, order[loc], -1)

E = np.load(EMB_NPY, mmap_mode="r")
Q = model.encode(test.q.tolist(), batch_size=EMBED_BATCH, convert_to_numpy=True,
                 normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

truth_sim = np.full(len(test), np.nan)
ok = row_of_truth >= 0
truth_sim[ok] = (Q[ok] * np.asarray(E[row_of_truth[ok]]).astype(np.float32)).sum(1)

cutoff_50  = sims[:, 49]      # similarity of the 50th-best retrieved
cutoff_500 = sims[:, -1]      # similarity of the 500th-best retrieved

not_embedded = ~ok
missed_500   = (ranks_prn == -1) & ok          # embedded but below rank 500
recoverable  = (ranks_prn > 50) & (ranks_prn <= 500)   # deeper K would catch these

print(f"of {len(test)} matchable queries:")
print(f"  true record not in embedded set at all : {int(not_embedded.sum()):>4}  "
      f"(cannot be retrieved: data coverage, not the model)")
print(f"  recoverable by K=500 but missed at 50  : {int(recoverable.sum()):>4}  "
      f"(deeper K alone recovers these)")
print(f"  missed even at K=500                   : {int(missed_500.sum()):>4}")

if missed_500.sum():
    gap = (cutoff_500 - truth_sim)[missed_500]
    print(f"\n  for the K=500 misses, how far below the cutoff was the truth?")
    print(f"    median gap : {np.nanmedian(gap):.3f}   (small = near miss, model almost had it)")
    print(f"    90th pct   : {np.nanpercentile(gap, 90):.3f}   (large = model badly wrong or bad label)")

    worst = test.loc[missed_500].copy()
    worst["truth_sim"] = truth_sim[missed_500]
    worst["cutoff500"] = cutoff_500[missed_500]
    worst["gap"] = cutoff_500[missed_500] - truth_sim[missed_500]
    print("\n  10 worst misses (largest gap), to check for label noise:")
    print(worst.sort_values("gap", ascending=False)
               .head(10)[["address", "matched", "truth_sim", "cutoff500", "gap"]]
               .to_string(index=False))

of 789 matchable queries:
  true record not in embedded set at all :    0  (cannot be retrieved: data coverage, not the model)
  recoverable by K=500 but missed at 50  :   44  (deeper K alone recovers these)
  missed even at K=500                   :   25

  for the K=500 misses, how far below the cutoff was the truth?
    median gap : 0.055   (small = near miss, model almost had it)
    90th pct   : 0.185   (large = model badly wrong or bad label)

  10 worst misses (largest gap), to check for label noise:
                                            address                                                              matched  truth_sim  cutoff500      gap
      Unit 846/2C Defried Avenue, Zetland, NSW 2017                                       2 DEFRIES AV, ZETLAND NSW 2017   0.393505   0.651396 0.257892
            1/21 Chester Street Camperdown NSW 2050                                   21 WINCHESTER ST, CARLTON NSW 2218   0.355883   0.556354 0.200471
                65B O'Reilly Ro

In [ ]:
# Cell 22: recall@K split by the shape of the true record: a single number, or
# a number range (such as 22-36) where the query gives the first, last or an
# interior number, or the whole range. Rows that cannot be classified go to
# "range_unclear" buckets and are printed. Writes
# OUT/stage1_recall_by_subset.tsv, which cell 23 reads.
import re
import numpy as np, pandas as pd
from pathlib import Path

assert "con" in globals() and "OUT" in globals(), "run cell 17 first"
PERQ = OUT / "stage1_recall_per_query.tsv"
assert PERQ.exists(), f"{PERQ} not found; run cell 20 first"

split = pd.read_csv(PERQ, sep="\t", dtype={"address": str, "truth_pid": str})
split["rank_principal"] = split.rank_principal.astype(int)
print(f"{len(split):,} evaluated queries read from {PERQ.name}")

# 0. recompute recall@50 from the file. It must equal the principal-pid
# recall@50 printed by cell 20, or the file belongs to a different run.
_r50 = float(((split.rank_principal > 0) & (split.rank_principal <= 50)).mean())
print(f"recall@50 recomputed from this file : {_r50:.4f}")
print("   must equal the principal-normalised recall@50 cell 20 printed.")
print("   If it does not, re-run cell 20 and then re-run this cell.\n")

# 1. the shape of each true record. nf is its first house number and nl its
# last (NULL for a single number), so "22-36 WALSH ST" gives nf=22, nl=36.
# One row per pid, preferring the street-number key, as in cell 20's truth join.
con.register("tp_df", pd.DataFrame({"pid": split.truth_pid.unique()}))
rng_df = con.sql("""
    SELECT t.pid,
           try_cast(k.k AS INTEGER)           AS nf,
           try_cast(g.number_last AS INTEGER) AS nl
    FROM tp_df t
    JOIN gnaf_key k ON k.pid = t.pid
    LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k
    QUALIFY row_number() OVER (PARTITION BY t.pid
        ORDER BY CASE WHEN k.k LIKE 'LOT %' THEN 1 ELSE 0 END, k.k) = 1
""").df()
RNG = {r.pid: (r.nf, r.nl) for r in rng_df.itertuples()}

# 2. strip everything in front of the house number. Two kinds of prefix:
#   CONSUMES_NUMBER  the word and the number after it both go
#                    ("Unit 9/120 Station St" -> "120 Station St")
#   NOISE_WORD       only the word goes, and the number after it is the house
#                    number ("House 68 St George's Tce" -> "68 ...")
# "Lot 36, 146 Parramatta Rd" has a lot prefix before a street number, so the
# prefix is stripped. "Lot 17 Creek Street" has only a lot number, which is a
# different numbering system, so house_number() returns "LOT" and does not guess.
CONSUMES_NUMBER = r"(?:UNIT|APT|APARTMENT|FLAT|LEVEL|LVL|SHOP|SUITE|STE|ROOM|OFFICE|WARD)"
NOISE_WORD      = r"(?:HOUSE|BUILDING|TOWER|BLOCK)"

def strip_occupancy(addr):
    s = str(addr).upper().replace("\u2013", "-").replace("\u2014", "-").strip()
    for _ in range(4):                      # repeat for stacked prefixes, "Unit 12/ Level 3"
        before = s
        s = re.sub(rf"^\s*{NOISE_WORD}\s+", "", s)
        s = re.sub(rf"^\s*{CONSUMES_NUMBER}\s*\d+[A-Z]?\s*[,/]?\s*", "", s)
        s = re.sub(r"^\s*LOT\s*\d+[A-Z]?\s*,\s*", "", s)      # lot as prefix
        s = re.sub(r"^\s*[A-Z]?\d+[A-Z]?\s*/\s*", "", s)      # 12/ and G01/
        if s == before:
            break
    return s

def house_number(addr):
    s = strip_occupancy(addr)
    if re.match(r"^\s*LOT\b", s):
        return "LOT"                        # lot number, not a house number
    m = re.match(r"\s*(\d+)", s)
    return int(m.group(1)) if m else None

def written_is_range(addr):
    # True when the query itself gives a range: "40-58", "2 - 24", "19A-27"
    return bool(re.match(r"\s*\d+[A-Z]?\s*-\s*\d+", strip_occupancy(addr)))

# 3. put each query in a bucket:
#   single          the true record is one number
#   range_written   the true record is a range and the query gives a range
#   range_first     the query gives the range's first number
#   range_last      the query gives the range's last number
#   range_interior  the query gives a number strictly inside the range
#   range_unclear_* could not be classified, printed in full below
def classify(row):
    nf, nl = RNG.get(row.truth_pid, (None, None))
    has_nf = nf is not None and not pd.isna(nf)
    has_nl = nl is not None and not pd.isna(nl)
    if not (has_nf and has_nl):
        return "single"
    lo, hi = int(nf), int(nl)
    if written_is_range(row.address):
        return "range_written"
    wn = house_number(row.address)
    if wn == "LOT":     return "range_unclear_lot"
    if wn is None:      return "range_unclear_noparse"
    if wn == lo:        return "range_first"
    if wn == hi:        return "range_last"
    if lo < wn < hi:    return "range_interior"
    return "range_unclear_outside"

split["subset"] = split.apply(classify, axis=1)
split["hit50"]  = (split.rank_principal > 0) & (split.rank_principal <= 50)

# 4. recall@K per bucket
KS = [1, 5, 10, 20, 50, 100, 200, 500]
def rec_at(r, k): return float(((r > 0) & (r <= k)).mean()) if len(r) else float("nan")
def wilson(p, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    d = 1 + z*z/n; c = p + z*z/(2*n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c - h)/d, (c + h)/d

ORDER = ["single", "range_first", "range_interior", "range_last",
         "range_written", "range_unclear_lot", "range_unclear_noparse",
         "range_unclear_outside"]
rows = []
for name in ORDER + ["ALL RANGE", "ALL"]:
    if   name == "ALL":       r = split.rank_principal
    elif name == "ALL RANGE": r = split[split.subset != "single"].rank_principal
    else:                     r = split[split.subset == name].rank_principal
    if len(r) == 0: continue
    rows.append({"subset": name, "n": len(r),
                 **{f"R@{k}": round(rec_at(r, k), 3) for k in KS}})
print("RECALL@K BY TRUTH-RECORD SHAPE")
print(pd.DataFrame(rows).to_string(index=False), "\n")

# 95% Wilson intervals for recall@50 on single, interior and all range rows
for name, mask in [("single",    split.subset == "single"),
                   ("interior",  split.subset == "range_interior"),
                   ("ALL RANGE", split.subset != "single")]:
    r = split.loc[mask, "rank_principal"]
    p = rec_at(r, 50); lo, hi = wilson(p, len(r))
    print(f"  recall@50, {name:10s}: {p:.4f}  [{lo:.3f}, {hi:.3f}]  (n={len(r)})")

# 5. print every unclassified row
unclear = split[split.subset.str.startswith("range_unclear")]
print(f"\nUNCLASSIFIED RANGE ROWS: {len(unclear)}  (eyeball all of them)")
for r in unclear.itertuples():
    nf, nl = RNG.get(r.truth_pid, (None, None))
    span = f"{int(nf)}-{int(nl)}" if (nf is not None and not pd.isna(nf)
                                      and nl is not None and not pd.isna(nl)) else "?"
    print(f"  {r.subset:22s} rank {r.rank_principal:>4}  covers {span:<12} {r.address[:56]}")

# 6. three examples per bucket, to check the rule by eye
print("\nSAMPLES PER BUCKET")
for name in ORDER[:5]:
    sub = split[split.subset == name].head(3)
    if not len(sub): continue
    print(f"\n  {name}:")
    for r in sub.itertuples():
        nf, nl = RNG.get(r.truth_pid, (None, None))
        span = f"{int(nf)}-{int(nl)}" if (nl is not None and not pd.isna(nl)) else f"{int(nf)}"
        print(f"    rank {r.rank_principal:>4}  covers {span:<12} {r.address[:56]}")

split.to_csv(OUT / "stage1_recall_by_subset.tsv", sep="\t", index=False)
print(f"\nwrote {OUT/'stage1_recall_by_subset.tsv'}")

789 evaluated queries read from stage1_recall_per_query.tsv
recall@50 recomputed from this file : 0.9125
   must equal the principal-normalised recall@50 cell 20 printed.
   If it does not, re-run cell 20 and then re-run this cell.

RECALL@K BY TRUTH-RECORD SHAPE
               subset   n   R@1   R@5  R@10  R@20  R@50  R@100  R@200  R@500
               single 634 0.596 0.727 0.793 0.850 0.929  0.956  0.975  0.976
          range_first  33 0.303 0.455 0.606 0.667 0.818  0.909  0.939  0.970
       range_interior  20 0.100 0.200 0.200 0.300 0.500  0.500  0.600  0.650
           range_last  54 0.333 0.574 0.630 0.796 0.907  0.981  0.981  0.981
        range_written  45 0.489 0.667 0.711 0.800 0.933  0.978  0.978  0.978
    range_unclear_lot   1 1.000 1.000 1.000 1.000 1.000  1.000  1.000  1.000
range_unclear_noparse   1 0.000 0.000 0.000 0.000 1.000  1.000  1.000  1.000
range_unclear_outside   1 0.000 1.000 1.000 1.000 1.000  1.000  1.000  1.000
            ALL RANGE 155 0.342 0.529 0.594

In [ ]:
# Cell 23: McNemar's exact test of recall@50, pairing each query's hit or miss
# under the baseline retriever (BASELINE_SUBSET, cell 2) with the current
# model, per subset, with a Holm correction across subsets. Queries missing
# from the baseline file are left out and counted.
import math
BASE_PERQ = BASELINE_SUBSET
CURR_PERQ = OUT / "stage1_recall_per_query.tsv"
SUBSETS   = OUT / "stage1_recall_by_subset.tsv"
K = 50

for p in [BASE_PERQ, CURR_PERQ, SUBSETS]:
    assert p.exists(), f"not found: {p}"

b_df = pd.read_csv(BASE_PERQ, sep="\t")[["address", "rank_principal"]]
c_df = pd.read_csv(CURR_PERQ, sep="\t")[["address", "rank_principal"]]
s_df = pd.read_csv(SUBSETS,  sep="\t")[["address", "subset"]]
m = (b_df.rename(columns={"rank_principal": "r_base"})
        .merge(c_df.rename(columns={"rank_principal": "r_new"}), on="address")
        .merge(s_df, on="address", how="left"))
_unpaired = len(c_df) - len(m)
if _unpaired:
    print(f"NOTE: {_unpaired} of {len(c_df)} current queries have no baseline row "
          "(the baseline file does not contain them). The test below pairs "
          f"the {len(m)} addresses both files share.")
assert len(m) > 0, "no addresses in common between the baseline and the current file"
m["hit_base"] = (m.r_base > 0) & (m.r_base <= K)
m["hit_new"]  = (m.r_new  > 0) & (m.r_new  <= K)

def exact(b, c):
    n = b + c
    if n == 0: return 1.0
    return min(1.0, 2*sum(math.comb(n, i) for i in range(min(b, c)+1))/2**n)

print(f"STAGE 1: BASELINE = {BASELINE_LABEL} vs CURRENT MODEL in {OUT}, recall@{K}, {len(m)} paired queries\n")
print(f"{'subset':16s} {'n':>4s} {'base':>7s} {'new':>7s} {'lost':>5s} {'resc':>5s} "
      f"{'net':>5s} {'p':>9s}")
rows = []
for name in ["single", "range_first", "range_last", "range_interior",
             "range_written", "ALL"]:
    g = m if name == "ALL" else m[m.subset == name]
    if not len(g): continue
    lost = int((g.hit_base & ~g.hit_new).sum())
    resc = int((~g.hit_base & g.hit_new).sum())
    p = exact(lost, resc)
    rows.append((name, p))
    print(f"{name:16s} {len(g):4d} {g.hit_base.mean():7.3f} {g.hit_new.mean():7.3f} "
          f"{lost:5d} {resc:5d} {resc-lost:+5d} {p:9.4f}")

print("\nHOLM CORRECTION across the subset tests")
sub = sorted([r for r in rows if r[0] != "ALL"], key=lambda t: t[1])
for i, (nm, p) in enumerate(sub):
    adj = min(1.0, p*(len(sub)-i))
    print(f"  {nm:16s} raw {p:.4f} -> adjusted {adj:.4f}  "
          + ("survives" if adj < 0.05 else "does not survive"))
print("\n'single' is the primary outcome. A non-significant result there means")
print("no significant change on single-number addresses.")

NOTE: 4 of 789 current queries have no baseline row (the baseline file does not contain them). The test below pairs the 785 addresses both files share.
STAGE 1: BASELINE = 85k model (results_labels_789_model_85k) vs CURRENT MODEL in output_stage1, recall@50, 785 paired queries

subset              n    base     new  lost  resc   net         p
single            634   0.920   0.929    14    20    +6    0.3915
range_first        31   0.806   0.806     1     1    +0    1.0000
range_last         54   0.889   0.907     0     1    +1    1.0000
range_interior     20   0.350   0.500     0     3    +3    0.2500
range_written      43   0.907   0.930     0     1    +1    1.0000
ALL               785   0.898   0.912    15    26   +11    0.1173

HOLM CORRECTION across the subset tests
  range_interior   raw 0.2500 -> adjusted 1.0000  does not survive
  single           raw 0.3915 -> adjusted 1.0000  does not survive
  range_first      raw 1.0000 -> adjusted 1.0000  does not survive
  range_last     

In [ ]:
# Cell 24: write OUT/eval_manifest.json, recording the test set (by sha1), the
# model weights (by modification time), the index size and recall@50, so a
# stale evaluation can be spotted.
_E = np.load(EMB_NPY, mmap_mode="r")
eval_manifest = dict(
    test_file=str(MISSPELLINGS_TSV), test_sha1=TEST_SHA1,
    n_matchable_evaluated=int(len(test)),
    recall_at_50_principal=round(float(((ranks_prn > 0) & (ranks_prn <= 50)).mean()), 4),
    model_dir=str(OUT / "model"), model_mtime=MODEL_MTIME,
    index_rows=int(_E.shape[0]), index_dtype=str(_E.dtype),
    per_query_file=str(OUT / "stage1_recall_per_query.tsv"),
    by_subset_file=str(OUT / "stage1_recall_by_subset.tsv"),
    baseline_file=str(BASELINE_SUBSET),
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
del _E
(OUT / "eval_manifest.json").write_text(json.dumps(eval_manifest, indent=2))
print(json.dumps(eval_manifest, indent=2))
print(f"\nwrote {OUT/'eval_manifest.json'}")

{
  "test_file": "output/misspellings.tsv",
  "test_sha1": "d48fdfe227fd6ef7828384f020e3042d18873101",
  "n_matchable_evaluated": 789,
  "recall_at_50_principal": 0.9125,
  "model_dir": "output_stage1/model",
  "model_mtime": 1788792286.0973577,
  "index_rows": 15901232,
  "index_dtype": "float32",
  "per_query_file": "output_stage1/stage1_recall_per_query.tsv",
  "by_subset_file": "output_stage1/stage1_recall_by_subset.tsv",
  "baseline_file": "results_labels_789_model_85k/stage1/stage1_recall_by_subset.tsv",
  "timestamp": "2026-09-08 00:12:53"
}

wrote output_stage1/eval_manifest.json
